# 🏗️ SAM 3 Building Counter — Inference Server

This notebook runs Meta's SAM 3 model on a **free Google Colab GPU** and exposes it as an API for the GIS Building Counter dashboard.

## Instructions
1. Make sure the runtime is set to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Click **Runtime → Run all** (or Ctrl+F9)
3. Copy the public URL printed at the bottom
4. Paste it into the dashboard **Settings** modal

> ⚠️ The free Colab session will stay active for ~12 hours. You can re-run this notebook anytime to get a new URL.

## Step 1 — Check GPU & Install Dependencies

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print('✅ GPU ready')
else:
    print('⚠️ No GPU detected — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Install cloudflared (free tunnel, no account needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm cloudflared-linux-amd64.deb
!cloudflared --version
print('✅ Cloudflared installed')

In [ ]:
%%capture
# Install server dependencies
!pip install -q flask flask-cors Pillow numpy

## Step 2 — Start the Inference Server

This cell starts the Flask server and creates a **free Cloudflare tunnel**.
The public URL will be printed below — copy it into the dashboard Settings.

> **Note:** If SAM 3 model loading fails (e.g. missing HuggingFace access), the server
> will still run in **simulation mode** so you can test the full dashboard pipeline.

In [ ]:
import io
import time
import json
import logging
import subprocess
import re
import random
from threading import Thread

import torch
import numpy as np
from PIL import Image
from flask import Flask, request, jsonify
from flask_cors import CORS

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- Model Loading ---
print('📦 Attempting to load SAM 3 model...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_LOADED = False

try:
    from sam3.build_sam import sam_model_registry
    model = sam_model_registry['sam3_hiera_large'](checkpoint=None)
    model = model.to(device).eval()
    print('✅ SAM 3 model loaded successfully')
    MODEL_LOADED = True
except Exception as e:
    print(f'⚠️ SAM 3 model not available: {e}')
    print('   Running in SIMULATION MODE — dashboard will work with test data')
    print('   (To enable real inference, install SAM 3 and authenticate with HuggingFace)')

# --- Flask Server ---
app = Flask(__name__)
CORS(app)


@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'healthy',
        'model_loaded': MODEL_LOADED,
        'device': device,
        'gpu_available': torch.cuda.is_available(),
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    })


@app.route('/predict', methods=['POST'])
def predict():
    start_time = time.time()

    if 'image' not in request.files:
        return jsonify({'error': 'No image provided'}), 400

    image_file = request.files['image']
    image = Image.open(image_file.stream).convert('RGB')
    width, height = image.size

    prompt = request.form.get('prompt', 'building')
    confidence_threshold = float(request.form.get('confidence_threshold', 0.5))
    min_area = request.form.get('min_area')
    max_area = request.form.get('max_area')
    if min_area: min_area = int(min_area)
    if max_area: max_area = int(max_area)

    logger.info(f'Processing {width}x{height} image with prompt: "{prompt}"')

    if MODEL_LOADED:
        try:
            img_array = np.array(image)
            from sam3.automatic_mask_generator import SamAutomaticMaskGenerator
            mask_generator = SamAutomaticMaskGenerator(model)
            masks = mask_generator.generate(img_array)

            boxes, scores, mask_data = [], [], []
            for m in masks:
                bbox = m['bbox']
                box = [bbox[0], bbox[1], bbox[0]+bbox[2], bbox[1]+bbox[3]]
                score = m.get('predicted_iou', m.get('stability_score', 0.8))
                area = bbox[2] * bbox[3]
                if score < confidence_threshold: continue
                if min_area and area < min_area: continue
                if max_area and max_area > 0 and area > max_area: continue
                boxes.append(box)
                scores.append(float(score))
                mask_data.append({'area': int(m['area']), 'size': [height, width]})

            processing_time = time.time() - start_time
            return jsonify({
                'masks': mask_data, 'boxes': boxes, 'scores': scores,
                'processing_time': round(processing_time, 2),
                'image_width': width, 'image_height': height,
            })
        except Exception as e:
            logger.error(f'Inference error: {e}')
            return jsonify({'error': str(e)}), 500
    else:
        # --- Simulation mode ---
        num = random.randint(8, 35)
        boxes, scores, mask_data = [], [], []
        for _ in range(num):
            x1 = random.randint(10, max(11, width - 80))
            y1 = random.randint(10, max(11, height - 80))
            w = random.randint(25, 90)
            h = random.randint(25, 90)
            s = random.uniform(0.45, 0.98)
            if s < confidence_threshold: continue
            area = w * h
            if min_area and area < min_area: continue
            if max_area and max_area > 0 and area > max_area: continue
            boxes.append([x1, y1, x1+w, y1+h])
            scores.append(round(s, 4))
            mask_data.append({'area': area, 'size': [height, width]})

        processing_time = time.time() - start_time
        return jsonify({
            'masks': mask_data, 'boxes': boxes, 'scores': scores,
            'processing_time': round(processing_time, 2),
            'image_width': width, 'image_height': height,
            'note': 'Simulated results — SAM 3 model not loaded',
        })


# --- Cloudflare Tunnel (free, no account needed) ---
def start_tunnel():
    time.sleep(3)
    try:
        proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--url', 'http://localhost:5000', '--no-autoupdate'],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        )
        for line in proc.stderr:
            line = line.decode()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                url = match.group()
                print()
                print('=' * 60)
                print('  🌐 SAM 3 API is live at:')
                print(f'  {url}')
                print()
                print('  📋 Copy this URL and paste it into')
                print('     the dashboard Settings modal (gear icon)')
                print('=' * 60)
                print()
                return
    except Exception as e:
        print(f'⚠️ Tunnel error: {e}')
        print('   Server running at http://localhost:5000 (local only)')


Thread(target=start_tunnel, daemon=True).start()
print('🚀 Starting SAM 3 inference server...')
print('   Waiting for Cloudflare tunnel URL (10-15 seconds)...\n')
app.run(host='0.0.0.0', port=5000, debug=False)